In [1]:
# import libraries
import gensim.downloader as api
from openai import OpenAI
import json
import time
from dotenv import load_dotenv
import os
import sys
from pathlib import Path

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../"
sys.path.append(str(base_path.resolve()))

from utils.data_augmentation import augmentation_non_entity, augmentation_entity, find_entity_span

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/maxweiland/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# load the data
with open("../01_data/annotations/annotations_no_augmentations_corrected.json", "r") as f:
    data_non_augmented = json.load(f)

In [3]:
load_dotenv()  # reads .env file
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# load the glove model
glove_model = api.load("glove-wiki-gigaword-50")

In [4]:
# Function for generative paraphrasing
def augmentation_generative(task, temp=0.5, model="gpt-4o-mini"):
    sentence = task["sentence"]
    entities = task["annotations"]

    if not entities:
        prompt = f"""
        You are given the following sentence:
        Sentence: {sentence}

        Paraphrase the sentence in natural English while preserving its meaning.
        Return only the paraphrased sentence.
        """
    else:
        # Construct a prompt that preserves entities
        prompt = f"""
        You are given the following sentence with annotations for social groups:
        Sentence: {sentence}
        Social groups: {entities}

        Paraphrase the sentence in natural English, preserving the meaning and the social groups exactly as they appear. 
        Return only the paraphrased sentence.
        """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp
    )

    paraphrased_sentence = response.choices[0].message.content.strip()
    new_task = find_entity_span(task, paraphrased_sentence)

    return new_task

In [5]:
# set the start time
start_time = time.time()

# empty list to store all augmentations
dataset_augmentations = []

# loop through all tasks
for i, task in enumerate(data_non_augmented):

    if i+1 == 1000:
        intermediate_time = time.time()
        print(f"1000 data points processed in {intermediate_time-start_time} seconds")
    elif i+1 == 2000:
        intermediate_time = time.time()
        print(f"2000 data points processed in {intermediate_time-start_time} seconds")
    elif i+1 == 3000:
        intermediate_time = time.time()
        print(f"3000 data points processed in {intermediate_time-start_time} seconds")
    elif i+1 == 4000:
        intermediate_time = time.time()
        print(f"4000 data points processed in {intermediate_time-start_time} seconds")

    # add an id column
    task_id = f"index_{i+1}"

    # get the original sentence and all annotations
    sentence = task["sentence"]
    annotations = sorted(task["annotations"], key=lambda x: x["start"])

    # apply synonym augmentations
    non_entity_aug = augmentation_non_entity(task, glove_model, aug_p=0.3)
    entity_aug = augmentation_entity(task, glove_model, aug_p_entity=1, aug_p_non_entitiy=0.3)

    # get generative paraphrasing from gpt model
    generative_aug = augmentation_generative(task, temp=0.5)

    # add all to the list
    dataset_augmentations.append({
        "id": task_id,
        "sentence": sentence,
        "annotations": annotations,
        "augmentations": [
            non_entity_aug,
            entity_aug,
            generative_aug
            ]
            })

end_time = time.time()
print(f"5000 data points processed in {end_time-start_time} seconds")

1000 data points processed in 1216.6134469509125 seconds
2000 data points processed in 2462.3994269371033 seconds
3000 data points processed in 3666.8307309150696 seconds
4000 data points processed in 4870.031866788864 seconds
5000 data points processed in 6152.081213951111 seconds


In [6]:
with open("../01_data/annotations/annotations_gpt_augmentations_corrected.json", "w") as f:
    json.dump(dataset_augmentations, f)